# Custom antenna and metric

Use an antenna model from the workspace `antennas` library and a metric specification from the workspace `metrics` library. The SDK reads these local files and uploads their contents to this in-memory planning scenario.

## 1. Connect to the planning service

In [ ]:
import pandas as pd
from cisei_planning_sdk import PlanningClient, PlanningProject, plot_network

client = PlanningClient(
    "http://network-service:8080",
    workspace="..",
    timeout=900,
)

## 2. Open the project

In [ ]:
project = PlanningProject("../projects/graph_planner_antenna_function_test")
scenario = client.open_project(project, planner="graph")

scenario.summary

## 3. Upload the custom configuration

`custom_omni.yaml` supplies the antenna model. The profile mapping applies that model to both antenna profiles in the project. `gain_aware_metric.toml` defines the metric used by the `antenna_test` interfaces. The files remain in the shared workspace libraries; their contents are active only in this server scenario.

In [ ]:
scenario.upload_configuration(
    antenna_library="custom_omni.yaml",
    antennas={
        "tower_pctel_boa9028": {"model_id": "custom_omni"},
        "leaf_pctel_boa9028": {"model_id": "custom_omni"},
    },
    metrics={"antenna_test": "gain_aware_metric.toml"},
)

## 4. Build the candidate link

In [ ]:
candidates = scenario.build_candidate_edges()

pd.DataFrame(candidates["candidate_edges"])

## 5. Compute the custom metric

The service extracts the geographic features and evaluates the uploaded metric using the uploaded antenna gains. The SDK saves the reusable features in this project's `features.json`.

In [ ]:
metrics = scenario.compute_metrics()

pd.DataFrame(metrics["metrics"])

## 6. Solve and evaluate the network

In [ ]:
solution = scenario.solve()

pd.DataFrame(solution["planned_edges"])

In [ ]:
evaluation = scenario.evaluate(rank_threshold=10)

evaluation["quality_summary"]

## 7. Save and inspect the result

Reload the saved project result before rendering its candidate and final views.

In [ ]:
scenario.save_result(rank_threshold=10)
saved_result = scenario.load_result()

In [ ]:
plot_network(saved_result, kind="candidate");

In [ ]:
plot_network(saved_result, kind="final");